# Lab 11-2 - Linear Regression

This lab covers the `sklearn` implementation of linear regression, which is straightforward and largely uses the same syntax as other algorithms we've studied.

In [ ]:
import pandas as pd
import math

Examples throughout the lab will use the Iowa City home sales data that we've seen in several of our previous labs. We have

- n = 621 houses
- p = 12 features used to predict

In [ ]:
# Read data
ic = pd.read_csv(filepath_or_buffer = "../data/IowaCityHomeSales.csv")

# Split training and testing sets
from sklearn.model_selection import train_test_split
train, test = train_test_split(ic, test_size = 0.2, random_state = 7)

# Create target and keep only the numeric predictors (though we could use one-hot encoding if we wanted to)
train_y = train['sale.amount']
train_X = train.select_dtypes(include = "number").drop(labels = 'sale.amount', axis = 1)

# Print n and p
print(train_X.shape)

## A) Linear Regression in `sklearn`

Linear regression is performed using the `LinearRegression()` function in `sklearn`. This function is compatible with pipelines and all the pre-processing tools we've been using:

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline 

# Simple pipeline using linear regression
pipe = Pipeline(steps = [
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

Compared to other algorithms, a benefit of linear regression is that we can easily determine the impact of each individual predictor by looking at its weight in the model's linear combination:

In [ ]:
# Fit the pipeline to the training data
lin_mod = pipe.fit(X = train_X, y = train_y)


# Print the estimated intercept (bias)
print(lin_mod.named_steps['model'].intercept_)

print("\n")


# print the model's 12 slope coefficients
print(lin_mod.named_steps['model'].coef_)

In [ ]:
# Now let's print the labeled slope coefficients
pd.Series(data = lin_mod.named_steps['model'].coef_, index = train_X.columns)

Note: 
- While standardization is not an essential step in linear regression, it has the benefit of making these estimated weights more directly comparable. That is, after standardizing each of these weights represents the estimated effect of a 1 standard deviation increase in the corresponding predictor on the model's outcome.
- You should also recognize that the model's estimated bias (intercept) is stored as a different attribute `.intercept_` than the slope coefficients for all our features `.coef_`

Linear regression does not have any tuning parameters, but we can still look at an example of it being used within a cross-validated grid search by exploring models with and without an intercept (bias):

In [ ]:
# Simple grid search
from sklearn.model_selection import GridSearchCV
params = {'model__fit_intercept': [True, False]}

grid_result = GridSearchCV(
    estimator = pipe, 
    param_grid = params, 
    cv = 5, 
    scoring = 'neg_root_mean_squared_error'
    ).fit(X = train_X, y = train_y)

# Using an intercept yields a lower RMSE
print(grid_result.best_params_)
print(grid_result.best_score_)

In [ ]:
# Recall this is how we extract the best cross-validated model from above for future use:
grid_result.best_estimator_

## B) Question 1

Suppose we'd like to model a home's *assessed value* rather than its sale price.

- **Part a)**: Create two objects, `train_X_assessed` and `train_y_assessed` where the latter records each home's assessed value and the former is a matrix containing the predictors: `area.living`, `bedrooms`, and `area.lot`.
- **Part b)**: Create a pipeline that includes min-max scaling followed by a Yeo-Johnson normalizing transformation on the model's predictors as preprocessing step priors to fitting a linear regression model in the pipeline's final step.
- **Part c)**: Use a cross-validated grid search with RMSE as the scoring metric to evaluate whether the use of a normalizing transformation on these predictors improves performance relative to an approach that doesn't involve any normalization.
- **Part d)**: Print the estimated weights of the best model in Part c). Is the predictor with the largest estimated weight the most important contributor in this model? *Hint*: think about the scale of each predictor and how that influences your interpretation of the corresponding weight parameter.

In [ ]:
# Code for Q1.a)
train_y_assessed = train['assessed']
train_X_assessed = train[['bedrooms','area.living','area.lot']]

In [ ]:
# Code for Q1.b)
from sklearn.preprocessing import MinMaxScaler, PowerTransformer

pipe_q1 = Pipeline(steps = [
    ('scaler', MinMaxScaler()),
    ('transformer', PowerTransformer(method = 'yeo-johnson')),
    ('model', LinearRegression())
])

In [ ]:
# Code for Q1.c)
params_q1 = {
    'transformer': [PowerTransformer(method = 'yeo-johnson'), 'passthrough']
}

grid_q1_result = GridSearchCV(
    estimator = pipe_q1, 
    param_grid = params_q1, 
    cv = 5, 
    scoring = 'neg_root_mean_squared_error'
    ).fit(X = train_X_assessed, y = train_y_assessed)

print(grid_q1_result.best_params_)
print(grid_q1_result.best_score_)

# No: Yeo-Johnson makes CV RMSE worse (about 54,032 vs 47,038), so 'passthrough' wins

In [ ]:
# Code for Q1.d)
print(dict(zip(train_X_assessed.columns, grid_q1_result.best_estimator_.named_steps['model'].coef_)))
# Yes - after min-max scaling, each coefficient is the change in predicted assessed value
# from the smallest to the largest observed value of that predictor, so the weights are
# directly comparable. (Only because the best model skipped the Yeo-Johnson step.)

## C) Question 2

Next you'll explore how linear regression compares with other methods we've learned. Throughout this question you should use RMSE as your scoring metric.

- **Part a)**: Using the objects you created in Part a) of Question 1, use a cross-validated grid search to find an appropriate k-nearest neighbors model for these data. Be sure to include any preprocessing steps you believe to be necessary for this approach. A satisfactory answer should explore at least a few different combinations of tuning parameters.
- **Part b)**: Using the objects you created in Part a) of Question 1, use a cross-validated grid search to find an appropriate `xgboost` model for these data. Be sure to include any preprocessing steps you believe to be necessary for this approach. A satisfactory answer should explore at least a few different combinations of tuning parameters.
- **Part c)**: Perform a final cross-validated grid search that compares the best models from Parts a) and b) of this question with the best model you identified in Question 1 (linear regression either with or without a normalizing transformation).
- **Part d)**: Find the RMSE of each model you considered in Part c) on the test set.

In [ ]:
# Code for Q2.a)
from sklearn.neighbors import KNeighborsRegressor
pipe_q2a_knn = Pipeline(steps = [
    ('scaler', MinMaxScaler()),
    ('model', KNeighborsRegressor())
])

params_q2a_knn = {
    'model__n_neighbors': [5,7,15],
    'model__weights': ['uniform', 'distance']
}

grid_q2a_knn_result = GridSearchCV(
    estimator = pipe_q2a_knn, 
    param_grid = params_q2a_knn, 
    cv = 5, 
    scoring = 'neg_root_mean_squared_error'
    ).fit(X = train_X_assessed, y = train_y_assessed)


print(grid_q2a_knn_result.best_params_)
print(grid_q2a_knn_result.best_score_)

In [ ]:
# Code for Q2.b)
import xgboost as xgb

pipe_q2b_xgb = Pipeline(steps = [
    ('scaler', MinMaxScaler()),
    ('model', xgb.XGBRegressor())
])

params_q2b_xgb = {
    'model__learning_rate': [0.01, 0.1, 0.3],
    'model__max_depth': [2,3,4]
    }

grid_q2b_xgb_result = GridSearchCV(
    estimator = pipe_q2b_xgb, 
    param_grid = params_q2b_xgb, 
    cv = 5, 
    scoring = 'neg_root_mean_squared_error'
    ).fit(X = train_X_assessed, y = train_y_assessed)

print(grid_q2b_xgb_result.best_params_)
print(grid_q2b_xgb_result.best_score_)

In [ ]:
# Code for Q2.c)
pipe_q2c = Pipeline(steps = [
    # Put XGB model as place holder for now:
    ('final_model', grid_q2b_xgb_result.best_estimator_)
])

model_list = {
    'final_model': [grid_q1_result.best_estimator_, grid_q2a_knn_result.best_estimator_, grid_q2b_xgb_result.best_estimator_]
}

grid_final = GridSearchCV(
    estimator = pipe_q2c, 
    param_grid = model_list, 
    cv = 5, 
    scoring = 'neg_root_mean_squared_error'
    ).fit(X = train_X_assessed, y = train_y_assessed)

# It appears XGBoost wins
print(grid_final.best_params_)
print(grid_final.best_score_)

In [ ]:
# Code for Q2.d)
from sklearn.metrics import mean_squared_error
test_y_assessed = test['assessed']
test_X_assessed = test[['bedrooms','area.living','area.lot']]

# Q1: linear regression test error
print("Linear regression RMSE: ", 
    math.sqrt(
        mean_squared_error(
            y_true = test_y_assessed, 
            y_pred = grid_q1_result.best_estimator_.predict(X = test_X_assessed)
        )
    )
)

# Q2.a) KNN test error 
print("KNN RMSE:", 
    math.sqrt(
        mean_squared_error(
            y_true = test_y_assessed, 
            y_pred = grid_q2a_knn_result.best_estimator_.predict(X = test_X_assessed)
        )
    )
)

# Q2.b) XGB test error 
print("XGB RMSE:", 
    math.sqrt(
        mean_squared_error(
            y_true = test_y_assessed, 
            y_pred = grid_q2b_xgb_result.best_estimator_.predict(X = test_X_assessed)
        )
    )
)

In [ ]:
# Compare cross-validated RMSE to the true test RMSE
finalists = {
    'Linear regression': grid_q1_result,
    'KNN': grid_q2a_knn_result,
    'XGBoost': grid_q2b_xgb_result
}

pd.DataFrame(data = {
    'Model': list(finalists.keys()),
    'CV RMSE': [-grid.best_score_ for grid in finalists.values()],
    'Test RMSE': [
        math.sqrt(mean_squared_error(
            y_true = test_y_assessed,
            y_pred = grid.best_estimator_.predict(X = test_X_assessed)
        ))
        for grid in finalists.values()
    ]
}).round(decimals = 0)

As expected (but not guaranteed) by the cross-validated score: XGBoost wins, then KNN, then linear regression.